<a href="https://colab.research.google.com/github/bored-shinigami2805/CVPR-SLM-fine-tuning/blob/main/SLM_train_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U "transformers>=4.46" "trl>=0.12" "peft>=0.13" \
    "datasets>=3.0" "accelerate>=1.0" bitsandbytes
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
import os, re, json, torch
from pathlib import Path
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
DATA_DIR  = "/content"                      # or a Google Drive folder (see cell 3)
SFT_PATH  = os.path.join(DATA_DIR, "sft_dataset.jsonl")
EVAL_PATH = os.path.join(DATA_DIR, "eval_benchmark.jsonl")
OUT_DIR   = "/content/qwen2p5-3b-cv-lora"
USE_4BIT  = True     # False on L4/A100 to train in bf16 (faster, more VRAM)

# The one fixed system prompt used by every row (kept here for the manual test cell).
SYS = ("You are a computer-vision research assistant. You have studied a specific set of "
       "computer-vision papers in depth. Answer accurately and concisely, grounded only in "
       "what you know. If a question is outside computer vision, say it is outside your domain. "
       "If it is about computer vision but not something you have studied, say you don't have "
       "that information rather than guessing.")

In [ ]:
if not (Path(SFT_PATH).exists() and Path(EVAL_PATH).exists()):
    try:
        from google.colab import files
        print("Select sft_dataset.jsonl and eval_benchmark.jsonl ...")
        up = files.upload()                      # saves into /content
    except Exception as e:
        print("Not in Colab or upload skipped:", e)

assert Path(SFT_PATH).exists(),  f"missing {SFT_PATH}"
assert Path(EVAL_PATH).exists(), f"missing {EVAL_PATH}"
n_train = sum(1 for _ in open(SFT_PATH, encoding="utf-8"))
n_eval  = sum(1 for _ in open(EVAL_PATH, encoding="utf-8"))
print(f"train rows: {n_train} | eval items: {n_eval}")

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

quant = None
if USE_4BIT:
    quant = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant,
    torch_dtype=torch.bfloat16, device_map="auto")
model.config.use_cache = False
print(model.config.model_type, "loaded |", "4-bit" if USE_4BIT else "bf16")

In [ ]:
peft_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

train_ds = load_dataset("json", data_files=SFT_PATH, split="train")

sft_cfg = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    packing=False,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model, args=sft_cfg, train_dataset=train_ds,
    peft_config=peft_cfg, processing_class=tok)
trainer.train()
trainer.save_model(OUT_DIR)
tok.save_pretrained(OUT_DIR)
ft_model = trainer.model      # PEFT model; adapter can be toggled for the baseline
print("adapter saved to", OUT_DIR)

In [ ]:
trainer.model.save_pretrained("/content/drive/MyDrive/qwen2p5-3b-cv-lora")

In [ ]:
ft_model.eval()

def gen(messages, max_new=256):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(ft_model.device)
    with torch.no_grad():
        out = ft_model.generate(**ids, max_new_tokens=max_new, do_sample=False,
                                pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def _norm(s):
    s = s.lower().replace("\u00d7","x").replace("\u2212","-").replace("\u207b","-")
    return re.sub(r"[\s,]", "", s)

def score_exact(gold, out):
    """Pass iff the exact numeric value (with unit/percent) appears in the output."""
    g = _norm(gold.split("(")[0]); o = _norm(out)
    if g and g in o:
        return True
    # fallback: compare the leading number token, requiring same digits
    gm = re.search(r"-?\d[\d.]*", g)
    return bool(gm) and gm.group(0) in o

def is_refusal(out):
    o = out.lower()
    return "outside" in o and ("domain" in o or "computer vision" in o)

def is_defer(out):
    o = out.lower()
    keys = ["don't have that information", "do not have that information",
            "not something i have", "i don't have", "i do not have",
            "isn't something i", "is not something i", "haven't studied",
            "have not studied"]
    return any(k in o for k in keys)

def rubric_hit(gold, out):
    toks = {t for t in re.split(r"[^a-z0-9.%]+", gold.lower()) if len(t) > 2}
    if not toks:
        return 0.0
    return sum(1 for t in toks if t in out.lower()) / len(toks)

def passed(item, out):
    sc = item["scoring"]
    if sc == "exact_match":     return score_exact(item["gold"], out)
    if sc == "refusal_correct": return is_refusal(out)
    if sc == "defer_correct":   return is_defer(out)
    if sc == "rubric":          return rubric_hit(item["gold"], out) >= 0.4  # heuristic; review CSV
    return False

In [ ]:
import pandas as pd
eval_items = [json.loads(l) for l in open(EVAL_PATH, encoding="utf-8")]

rows = []
for it in eval_items:
    msgs = it["messages"]
    with ft_model.disable_adapter():          # base model
        base_out = gen(msgs)
    ft_out = gen(msgs)                         # fine-tuned
    rows.append({
        "id": it["id"], "category": it["category"], "scoring": it["scoring"],
        "question": msgs[-1]["content"], "gold": it["gold"],
        "base_out": base_out, "ft_out": ft_out,
        "base_pass": bool(passed(it, base_out)), "ft_pass": bool(passed(it, ft_out)),
    })

df = pd.DataFrame(rows)
report = (df.groupby("category")
            .agg(n=("id","size"),
                 base_pct=("base_pass", lambda s: round(100*s.mean(),1)),
                 ft_pct=("ft_pass",   lambda s: round(100*s.mean(),1)))
            .reset_index())
print(report.to_string(index=False))
print("\nOVERALL  base %.1f%%  ->  fine-tune %.1f%%" % (
    100*df.base_pass.mean(), 100*df.ft_pass.mean()))
df.to_csv("/content/eval_results.csv", index=False)
print("per-item results written to /content/eval_results.csv")

In [ ]:
for q in ["What backbone does ICRDrag use?",
          "What IoU does ShutterMuse reach on the photographer-side task?",
          "What FID did ICRDrag report on the PRD benchmark?",   # false premise -> should defer
          "What is a good recipe for lasagna?"]:                  # OOD -> should refuse
    msgs = [{"role":"system","content":SYS},{"role":"user","content":q}]
    print("Q:", q)
    print("A:", gen(msgs), "\n")
    # In your gen() function, make sure these are set:
model.generation_config.max_new_tokens = 200
model.generation_config.do_sample = False  # greedy is faster than sampling
model.generation_config.use_cache = True   # KV cache on

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/qwen2p5-3b-cv-lora /content/drive/MyDrive/